## Importing all libraries needed to begin this project

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd

os.makedirs('output', exist_ok=True)
print('Libraries loaded successfully')

Libraries loaded successfully


## Loading the data into conda environment

In [2]:
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

orders    = pd.read_csv('data/olist_orders_dataset.csv',       parse_dates=date_cols)
items     = pd.read_csv('data/olist_order_items_dataset.csv')
customers = pd.read_csv('data/olist_customers_dataset.csv')
reviews   = pd.read_csv('data/olist_order_reviews_dataset.csv')
payments  = pd.read_csv('data/olist_order_payments_dataset.csv')

print(f'Orders loaded   : {len(orders):,}')
print(f'Customers loaded: {len(customers):,}')
print(f'Items loaded    : {len(items):,}')

Orders loaded   : 99,441
Customers loaded: 99,441
Items loaded    : 112,650


## Load SQLite into conda environment

In [3]:
conn = sqlite3.connect(':memory:')

orders.to_sql('orders',       conn, index=False, if_exists='replace')
items.to_sql('order_items',   conn, index=False, if_exists='replace')
customers.to_sql('customers', conn, index=False, if_exists='replace')
reviews.to_sql('reviews',     conn, index=False, if_exists='replace')
payments.to_sql('payments',   conn, index=False, if_exists='replace')

print('All tables loaded into SQLite')

All tables loaded into SQLite


## Identifying Top 8 regions be order volume

In [4]:
top_states = pd.read_sql("""
    SELECT c.customer_state, COUNT(DISTINCT o.order_id) AS order_count
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_state
    ORDER BY order_count DESC
    LIMIT 8
""", conn)

states_list = "', '".join(top_states['customer_state'].tolist())
print('Top 8 regions selected:')
print(top_states.to_string(index=False))

Top 8 regions selected:
customer_state  order_count
            SP        41746
            RJ        12852
            MG        11635
            RS         5466
            PR         5045
            SC         3637
            BA         3380
            DF         2140


## Generating six KPI queries

In [5]:
df_ontime = pd.read_sql(f"""
    SELECT
        c.customer_state AS region,
        COUNT(*) AS total_delivered,
        SUM(CASE WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
            THEN 1 ELSE 0 END) AS on_time_count,
        ROUND(100.0 * SUM(CASE WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
            THEN 1 ELSE 0 END) / COUNT(*), 2) AS on_time_rate_pct
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND c.customer_state IN ('{states_list}')
    GROUP BY c.customer_state
""", conn)

df_delivery = pd.read_sql(f"""
    SELECT
        c.customer_state AS region,
        ROUND(AVG(
            julianday(o.order_delivered_customer_date) -
            julianday(o.order_purchase_timestamp)
        ), 2) AS avg_delivery_days
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND c.customer_state IN ('{states_list}')
    GROUP BY c.customer_state
""", conn)

df_freight = pd.read_sql(f"""
    SELECT
        c.customer_state AS region,
        ROUND(AVG(oi.freight_value), 2) AS avg_freight_cost
    FROM orders o
    JOIN customers c   ON o.customer_id  = c.customer_id
    JOIN order_items oi ON o.order_id    = oi.order_id
    WHERE c.customer_state IN ('{states_list}')
    GROUP BY c.customer_state
""", conn)

df_cancel = pd.read_sql(f"""
    SELECT
        c.customer_state AS region,
        COUNT(*) AS total_orders,
        SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
        ROUND(100.0 * SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END)
              / COUNT(*), 2) AS cancel_rate_pct
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE c.customer_state IN ('{states_list}')
    GROUP BY c.customer_state
""", conn)

df_review = pd.read_sql(f"""
    SELECT
        c.customer_state AS region,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN reviews r   ON o.order_id   = r.order_id
    WHERE c.customer_state IN ('{states_list}')
    GROUP BY c.customer_state
""", conn)

df_revenue = pd.read_sql(f"""
    SELECT
        c.customer_state AS region,
        ROUND(SUM(p.payment_value) / COUNT(DISTINCT o.order_id), 2) AS avg_revenue_per_order,
        ROUND(SUM(p.payment_value), 2) AS total_revenue
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN payments p  ON o.order_id   = p.order_id
    WHERE c.customer_state IN ('{states_list}')
    GROUP BY c.customer_state
""", conn)

print('All 6 KPI queries executed successfully')

All 6 KPI queries executed successfully


## Merging all of my KPIs into a dashboard

In [6]:
kpi = df_ontime[['region', 'total_delivered', 'on_time_rate_pct']]
for df, cols in [
    (df_delivery, ['region', 'avg_delivery_days']),
    (df_freight,  ['region', 'avg_freight_cost']),
    (df_cancel,   ['region', 'total_orders', 'cancel_rate_pct']),
    (df_review,   ['region', 'avg_review_score']),
    (df_revenue,  ['region', 'avg_revenue_per_order', 'total_revenue']),
]:
    kpi = kpi.merge(df[cols], on='region')

print('=== KPI Summary by Region ===')
kpi

=== KPI Summary by Region ===


,region,total_delivered,on_time_rate_pct,avg_delivery_days,avg_freight_cost,total_orders,cancel_rate_pct,avg_review_score,avg_revenue_per_order,total_revenue
0,BA,3256,85.96,19.34,26.36,3380,0.47,3.86,182.44,616645.82
1,DF,2080,92.93,12.97,21.04,2140,0.33,4.06,165.95,355141.08
2,MG,11354,94.39,12.01,20.63,11635,0.55,4.14,160.92,1872257.26
3,PR,4923,95.00,11.99,20.53,5045,0.44,4.18,160.78,811156.38
4,RJ,12350,86.53,15.31,20.96,12852,0.67,3.87,166.85,2144379.69
5,RS,5344,92.85,15.30,21.74,5466,0.46,4.13,162.99,890898.54
6,SC,3546,90.24,14.95,21.47,3637,0.52,4.07,171.32,623086.43
7,SP,40494,94.11,8.76,15.15,41746,0.78,4.17,143.69,5998226.96


## KPI Insights

In [7]:
best_ontime  = kpi.loc[kpi['on_time_rate_pct'].idxmax(), 'region']
worst_ontime = kpi.loc[kpi['on_time_rate_pct'].idxmin(), 'region']
cheapest     = kpi.loc[kpi['avg_freight_cost'].idxmin(), 'region']
most_revenue = kpi.loc[kpi['total_revenue'].idxmax(), 'region']

print('=== Executive Insights ===')
print(f'Highest on-time delivery rate : {best_ontime}  ({kpi.loc[kpi.region==best_ontime, "on_time_rate_pct"].values[0]}%)')
print(f'Lowest  on-time delivery rate : {worst_ontime} ({kpi.loc[kpi.region==worst_ontime, "on_time_rate_pct"].values[0]}%)')
print(f'Lowest avg freight cost       : {cheapest}')
print(f'Highest total revenue region  : {most_revenue}')
print(f'\nAvg on-time rate across regions: {kpi["on_time_rate_pct"].mean():.1f}%')
print(f'Avg delivery time across regions: {kpi["avg_delivery_days"].mean():.1f} days')

=== Executive Insights ===
Highest on-time delivery rate : PR  (95.0%)
Lowest  on-time delivery rate : BA (85.96%)
Lowest avg freight cost       : SP
Highest total revenue region  : SP

Avg on-time rate across regions: 91.5%
Avg delivery time across regions: 13.8 days


## Monthly trend data

In [8]:
monthly = pd.read_sql(f"""
    SELECT
        c.customer_state AS region,
        SUBSTR(o.order_purchase_timestamp, 1, 7) AS month,
        COUNT(DISTINCT o.order_id) AS order_volume,
        ROUND(100.0 * SUM(CASE
            WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
            THEN 1 ELSE 0 END) /
            NULLIF(SUM(CASE
                WHEN o.order_status = 'delivered'
                 AND o.order_delivered_customer_date IS NOT NULL
                THEN 1 ELSE 0 END), 0), 2) AS on_time_rate_pct,
        ROUND(AVG(oi.freight_value), 2) AS avg_freight_cost
    FROM orders o
    JOIN customers c    ON o.customer_id = c.customer_id
    LEFT JOIN order_items oi ON o.order_id = oi.order_id
    WHERE c.customer_state IN ('{states_list}')
    GROUP BY c.customer_state, SUBSTR(o.order_purchase_timestamp, 1, 7)
    ORDER BY region, month
""", conn)

print(f'Monthly trend rows: {len(monthly):,}')
monthly.head(10)

Monthly trend rows: 177


,region,month,order_volume,on_time_rate_pct,avg_freight_cost
0,BA,2016-10,4,100.00,25.32
1,BA,2017-01,25,96.55,18.51
2,BA,2017-02,59,91.94,24.57
3,BA,2017-03,91,91.26,27.51
4,BA,2017-04,93,84.21,25.94
5,BA,2017-05,127,95.14,24.02
6,BA,2017-06,106,91.80,22.64
7,BA,2017-07,155,94.81,23.68
8,BA,2017-08,158,97.09,24.83
9,BA,2017-09,170,79.79,22.92


## Exporting results into a Tableau dashboard

In [9]:
kpi.to_csv('output/kpi_by_region.csv', index=False)
monthly.to_csv('output/kpi_monthly_trend.csv', index=False)
conn.close()

print('Exported:')
print('  output/kpi_by_region.csv')
print('  output/kpi_monthly_trend.csv')

Exported:
  output/kpi_by_region.csv      <- KPI snapshot (Tableau primary source)
  output/kpi_monthly_trend.csv  <- Monthly trends (Tableau secondary source)
